# CNN Crop and Weed Classification using VGG16

**Nalacherla Shravan Kumar • 23EG110C43 • Roll Number 43**

Individual study project. ImageNet transfer learning, local Jupyter, 12 species mapped to Crop (0) and Weed (1). Full training is the default. Smoke metrics are execution checks only.

## Environment verification

Select the Crop Weed VGG16 kernel. Dependencies must be installed in .venv.

In [ ]:
from pathlib import Path
import sys, os
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'scripts/core.py').exists()), None)
assert ROOT is not None, 'Start Jupyter from this repository or its notebooks folder.'
sys.path.insert(0, str(ROOT/'scripts'))
from verify_environment import verify
environment = verify()


## Configuration

Batch 16 by default. Set SMOKE_TEST=True for one quick epoch, or reduce batch size to 2 for low-memory laptops.

In [ ]:
from core import Config, Experiment, predict_image
SMOKE_TEST = False
config = Config(batch_size=16, smoke_test=SMOKE_TEST)
experiment = Experiment(config)

## Dataset location

Download first with python scripts/download_dataset.py. Nested extraction is supported.

In [ ]:
print(config.data_dir)

## Species validation

Decode every image and confirm the 12 expected species and 5,539 exact V2 counts.

In [ ]:
from dataset import SPECIES, CROPS
df = experiment.explore()
display(df.groupby('species').size().to_frame('images'))

## Binary label mapping

Crop: Maize, Common Wheat, Sugar Beet. The other nine species are Weed.

In [ ]:
display(df[['species','label']].drop_duplicates().sort_values('label'))
assert set(df.label) == {0, 1}

## Dataset exploration

Inspect paths and image counts. Images use controlled backgrounds and may share plant identity across growth stages.

In [ ]:
display(df.head())
print(df.shape)

## Class distribution

Class imbalance motivates training-only balanced weights.

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(experiment.output/'class_distribution.png')))

## Sample images

Inspect representative original RGB images.

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(experiment.output/'sample_images.png')))

## Stratified splitting

Seed 42. Split before augmentation, 70% training, 15% validation, 15% test. Assert no overlapping paths. Save dataset_splits.csv.

In [ ]:
splits = experiment.prepare()
display(splits.groupby(['split','label']).size().unstack())

## TensorFlow data pipeline

Load and resize raw RGB pixels to 224×224. Keep values in 0–255. A small prefetch queue limits memory.

In [ ]:
images, labels = next(iter(experiment.train_ds))
print(images.shape, labels.shape, float(images.numpy().max()))

## Data augmentation

Training-only horizontal flip, small rotation, zoom and translation. Keras disables random transforms during evaluation.

In [ ]:
from core import augmentation
augmentation(config.seed).summary()

## Class weights

Balanced weights are calculated only from training labels.

In [ ]:
print(experiment.weights)

## VGG16 architecture

ImageNet VGG16 without top; preprocessing after augmentation; global average pooling; Dense 128 ReLU; Dropout 0.5; one sigmoid.

In [ ]:
model = experiment.build()

## Phase 1 training

Freeze all VGG16 layers. Adam 0.001, binary cross-entropy, at most 10 epochs. Checkpoint, early stopping and plateau reduction monitor validation loss.

In [ ]:
head_history = experiment.train_head()

## Phase 2 fine-tuning

Unfreeze only block5, recompile Adam at 0.00001 and train up to five additional epochs. Smoke mode validates flags without a second epoch.

In [ ]:
fine_history = experiment.fine_tune()

## Combined training history

Combine both phases and save the measured epoch history.

In [ ]:
history = experiment.plot_history()
display(history)

## Accuracy graph

Compare training and validation accuracy by epoch.

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(experiment.output/'accuracy_graph.png')))

## Loss graph

Validation loss selects the best checkpoint across both phases.

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(experiment.output/'loss_graph.png')))

## Test evaluation

Reload the best checkpoint. Verify save/reload prediction equivalence. Evaluate the held-out test images once.

In [ ]:
results = experiment.evaluate()
print('Test loss:', results['test_loss'])
print('Test accuracy:', results['test_accuracy'])
print('AUC:', results['auc'])

## Confusion matrix

Rows are actual labels; columns are predicted labels. Crop=0, Weed=1.

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(experiment.output/'confusion_matrix.png')))

## Classification report

Both classes include precision, recall, F1 and support. Undefined precision is reported as zero.

In [ ]:
import pandas as pd
display(pd.read_csv(experiment.output/'classification_report.csv', index_col=0))

## Sample predictions

The sigmoid gives Weed probability, thresholded at 0.5.

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(experiment.output/'sample_predictions.png')))

## Misclassified examples

Inspect errors before drawing conclusions about field use.

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(experiment.output/'misclassified_examples.png')))

## Single-image prediction

Pass an RGB image path. Saved model includes VGG16 preprocessing.

In [ ]:
example_path = experiment.test.iloc[0].path
predict_image(experiment.model, example_path)

## Saving model and results

All outputs use repository-relative directories. Smoke results remain separate. Rebuild report and presentation after a full run.

In [ ]:
print('Best model:', experiment.model_path.relative_to(ROOT))
print('Results:', (experiment.output/'results_summary.json').relative_to(ROOT))
assert experiment.model_path.exists()
assert (experiment.output/'test_predictions.csv').exists()